# THESIS-003 — Workload Implementation
**Story Points:** 2 | **Status:** DONE

Workload image built from `src/Containerfile` (Python 3.13, dagster-k8s 0.28.22).
Image loaded into Kind local registry. SHA-256 hashing loop confirmed, ~30s/run.


## Acceptance Criteria
- [x] `workload_job.py` with `WORKLOAD_DURATION_SECONDS` env var
- [x] Containerfile that packages the workload for K8s
- [x] Workload produces deterministic CPU load
- [x] ~30-second default duration per job
- [x] Iteration count logged
- [ ] Single-job execution time within 5% between VM and K8s (baseline L1)

## Verify workload source

In [ ]:
import subprocess
r = subprocess.run(['cat', '../src/workload/workload_job.py'], capture_output=True, text=True)
for line in r.stdout.splitlines():
    if any(k in line for k in ['WORKLOAD_DURATION', 'sha256', 'def cpu_burn', 'hashlib']):
        print(line)

## Trigger a single K8s run and verify timing

In [ ]:
import subprocess, time
r = subprocess.run(
    ['python3', '../scripts/trigger_dagster_runs.py',
     '--host', 'localhost', '--port', '3001',
     '--level', '1', '--rep', '99', '--env', 'k8s', '--no-wait'],
    capture_output=True, text=True
)
print(r.stdout[-1000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
else:
    print('Run triggered. Watch pods:')
    time.sleep(2)
    r2 = subprocess.run(['kubectl', 'get', 'pods', '-n', 'dagster'], capture_output=True, text=True)
    print(r2.stdout)